# LeetCode 1506 — Find Root of N-Ary Tree bằng Python

Link đề: https://leetcode.com/problems/find-root-of-n-ary-tree/ · Độ khó: **Medium** · Chủ đề: Hash Table, Bit Manipulation, Tree, Depth-First Search

> **Ghi chú Premium:** đây là bài Premium 🔒. Đề bài và chữ ký trong notebook được viết theo hiểu biết về bài, lấy từ trang lời giải doocs (https://leetcode.doocs.org/en/lc/1506/). Nếu có tài khoản Premium, bạn nên đối chiếu lại đề gốc.

Skeleton LeetCode (theo lời giải Python của doocs):

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val=None, children=None):
        self.val = val
        self.children = children if children is not None else []
"""

class Solution:
    def findRoot(self, tree: list['Node']) -> 'Node':
        ...
```

## 1. Mục tiêu học

1. Hiểu cách LeetCode tuần tự hoá **cây N-phân** (level-order, `null` ngăn nhóm con) và tự viết hàm dựng/tuần tự hoá để test.
2. Nhận ra tính chất then chốt: **gốc là nút duy nhất không là con của nút nào** (bậc vào = 0).
3. Viết được 4 cách: brute force `O(n²)`, tập hợp các con `O(n)` bộ nhớ, **tổng giá trị** và **XOR** với bộ nhớ `O(1)` (follow-up).
4. Hiểu vì sao "mỗi giá trị không phải gốc xuất hiện đúng 2 lần" khiến phép trừ tổng / XOR triệt tiêu chúng.
5. Biết test đúng kiểu bài: xáo trộn danh sách nút (seed cố định) và kiểm tra trả về **đúng đối tượng gốc** (`is`), không chỉ đúng giá trị.

## 2. Đề bài & ví dụ

Cho **tất cả** các nút của một cây N-phân dưới dạng mảng `tree` gồm các đối tượng `Node`, **thứ tự tuỳ ý**. Mỗi nút có giá trị **duy nhất**. Hãy trả về **nút gốc** (chính đối tượng `Node` đó).

**Cách chấm (custom testing):** driver đọc chuỗi tuần tự hoá → dựng cây → đặt mọi `Node` vào một mảng theo thứ tự bất kỳ → gọi `findRoot` → tuần tự hoá nút trả về và so với input.

**Ví dụ 1:** `tree = [1,null,3,2,4,null,5,6]` → Output `[1,null,3,2,4,null,5,6]`. Mảng truyền vào có thể là `[Node(5),Node(4),Node(3),Node(6),Node(2),Node(1)]`; ta phải trả `Node(1)`.

**Ví dụ 2:** `tree = [1,null,2,3,4,5,null,null,6,7,null,8,null,9,10,null,null,11,null,12,null,13,null,null,14]` → output giống input.

**Ràng buộc:** tổng số nút trong `[1, 5·10⁴]`; mỗi nút có giá trị duy nhất.

**Follow-up:** giải được với bộ nhớ phụ `O(1)` và thời gian tuyến tính không?

In [1]:
import random
import time
from collections import deque

class Node:
    def __init__(self, val=None, children=None):
        self.val = val
        self.children = children if children is not None else []
    def __repr__(self):
        return f"Node({self.val})"

ex1 = [1, None, 3, 2, 4, None, 5, 6]
ex2 = [1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None,
       11, None, 12, None, 13, None, None, 14]
examples = [ex1, ex2]
for e in examples:
    print(e)

[1, None, 3, 2, 4, None, 5, 6]
[1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None, 11, None, 12, None, 13, None, None, 14]


Các hàm phụ tự dựng để test (driver giả lập):
- `build_nary(data)`: dựng cây từ dạng level-order có `None` phân nhóm, trả về gốc.
- `serialize(root)`: chiều ngược lại.
- `all_nodes(root)`: thu mọi nút (BFS).
- `make_input(data, seed)`: dựng cây, lấy mọi nút, **xáo trộn với seed cố định**, trả `(gốc thật, danh sách đã xáo)`.

In [2]:
def build_nary(data):
    if not data:
        return None
    root = Node(data[0])
    q = deque([root])
    i = 2                                  # bỏ root và None ngay sau root
    while q and i <= len(data):
        cur = q.popleft()
        while i < len(data) and data[i] is not None:
            ch = Node(data[i])
            cur.children.append(ch)
            q.append(ch)
            i += 1
        i += 1                             # bỏ None kết thúc nhóm con của cur
    return root

def serialize(root):
    if root is None:
        return []
    out = [root.val, None]
    q = deque([root])
    while q:
        cur = q.popleft()
        for ch in cur.children:
            out.append(ch.val)
            q.append(ch)
        out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def all_nodes(root):
    res, q = [], deque([root])
    while q:
        cur = q.popleft()
        res.append(cur)
        q.extend(cur.children)
    return res

def make_input(data, seed=0):
    root = build_nary(data)
    nodes = all_nodes(root)
    random.Random(seed).shuffle(nodes)
    return root, nodes

for e in examples:
    r, nodes = make_input(e, seed=1506)
    print(serialize(r) == e, nodes[:8], "...")

True [Node(6), Node(2), Node(5), Node(1), Node(4), Node(3)] ...
True [Node(7), Node(12), Node(14), Node(11), Node(9), Node(13), Node(5), Node(6)] ...


## 3. Kiến thức nền

### 3.1 Cây N-phân trong Python

Mỗi nút giữ `val` và `children` (list các nút con). Lưu ý định nghĩa dùng `children if children is not None else []` để **mỗi nút có list riêng** (tránh bẫy tham số mặc định `children=[]`).

In [3]:
a = Node(1, [Node(3, [Node(5), Node(6)]), Node(2), Node(4)])
print(a, a.children, a.children[0].children)
print(serialize(a))

x, y = Node(7), Node(8)
x.children.append(Node(9))
print("y.children vẫn rỗng:", y.children)

Node(1) [Node(3), Node(2), Node(4)] [Node(5), Node(6)]
[1, None, 3, 2, 4, None, 5, 6]
y.children vẫn rỗng: []


### 3.2 So sánh đối tượng: `is` vs `==`, và `id()`

`is` kiểm tra **cùng một đối tượng**. Lớp `Node` không định nghĩa `__eq__`, nên `==` cũng so theo danh tính; hai nút cùng giá trị nhưng khác đối tượng là khác nhau. Đề bắt trả **đúng đối tượng** trong mảng.

In [4]:
p = Node(1)
q = Node(1)
print(p is q, p == q, p is p)
print("id khác nhau:", id(p) != id(q))

False False True
id khác nhau: True


### 3.3 `set` chứa đối tượng

Đối tượng tự định nghĩa (không có `__eq__`/`__hash__` riêng) được băm theo danh tính → có thể bỏ vào `set` và kiểm tra `in` trong `O(1)` trung bình.

In [5]:
s = {p}
print(p in s, q in s)
vals = {n.val for n in [p, q]}
print("tập giá trị:", vals)

True False
tập giá trị: {1}


### 3.4 Phép XOR `^`

Tính chất: `a ^ a = 0`, `a ^ 0 = a`, giao hoán và kết hợp. Vậy XOR một dãy trong đó mọi số xuất hiện **chẵn** lần trừ một số xuất hiện **lẻ** lần sẽ cho đúng số đó.

In [6]:
seq = [5, 3, 9, 3, 5]
x = 0
for v in seq:
    x ^= v
    print(f"^ {v:2d} -> {x:2d} (nhị phân {x:04b})")
print("số xuất hiện lẻ lần:", x)

^  5 ->  5 (nhị phân 0101)
^  3 ->  6 (nhị phân 0110)
^  9 -> 15 (nhị phân 1111)
^  3 -> 12 (nhị phân 1100)
^  5 ->  9 (nhị phân 1001)
số xuất hiện lẻ lần: 9


### 3.5 `next(generator)` để lấy phần tử đầu tiên thoả điều kiện

`next(n for n in tree if n.val == x)` dừng ngay khi tìm thấy. Có thể truyền giá trị mặc định: `next(..., None)`.

In [7]:
nums = [4, 8, 15, 16, 23]
print(next(v for v in nums if v > 10))
print(next((v for v in nums if v > 100), None))

15
None


## 4. Ý tưởng nền tảng / trực giác

Mỗi cạnh của cây nối **cha → con**. Mọi nút trừ gốc có **đúng một cha**, gốc không có cha. Vì vậy:

- Nếu ta đánh dấu mọi nút xuất hiện trong `children` của ai đó, nút **không bị đánh dấu** là gốc.

Tính tay Ví dụ 1 (cây: 1 → {3, 2, 4}, 3 → {5, 6}):

| nút trong mảng | children | đánh dấu |
|---|---|---|
| 5 | — | |
| 4 | — | |
| 3 | 5, 6 | 5, 6 |
| 6 | — | |
| 2 | — | |
| 1 | 3, 2, 4 | 3, 2, 4 |

Các nút bị đánh dấu: {2, 3, 4, 5, 6}. Không bị đánh dấu: **1** → gốc.

**Đếm số lần xuất hiện:** liệt kê mỗi nút một lần (như phần tử của mảng) và mỗi lần nó là con → giá trị không phải gốc xuất hiện **2 lần**, gốc **1 lần**. Đây là chìa khoá cho cách tổng và XOR.

In [8]:
from collections import Counter
root, nodes = make_input(ex1, seed=3)
cnt = Counter()
for nd in nodes:
    cnt[nd.val] += 1
    for ch in nd.children:
        cnt[ch.val] += 1
print("thứ tự mảng:", nodes)
print("số lần xuất hiện:", dict(cnt))
print("giá trị xuất hiện 1 lần:", [v for v, c in cnt.items() if c == 1])

thứ tự mảng: [Node(1), Node(2), Node(4), Node(6), Node(5), Node(3)]
số lần xuất hiện: {1: 1, 3: 2, 2: 2, 4: 2, 6: 2, 5: 2}
giá trị xuất hiện 1 lần: [1]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi nút ứng viên `cand`, duyệt mọi nút khác xem `cand` có nằm trong `children` của nút đó không. Ứng viên không là con của ai là gốc.

In [9]:
def brute_root(tree):
    for cand in tree:
        is_child = False
        for nd in tree:
            for ch in nd.children:
                if ch is cand:
                    is_child = True
                    break
            if is_child:
                break
        if not is_child:
            return cand
    return None

for e in examples:
    root, nodes = make_input(e, seed=11)
    got = brute_root(nodes)
    print(got, got is root, serialize(got) == e)

Node(1) True True
Node(1) True True


**Độ phức tạp:** mỗi ứng viên quét toàn bộ các cạnh (`n − 1` cạnh) → Time `O(n²)`, Space `O(1)`. Với `n = 5·10⁴` là ~`2.5·10⁹` phép — quá chậm.

**Nút thắt:** ta quét lại toàn bộ cạnh cho **mỗi** ứng viên. Chỉ cần quét cạnh **một lần** và ghi nhớ ai là con.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** "Gốc là nút có bậc vào 0" → chỉ cần một lượt quét mọi cạnh.

- **Ghi nhớ bằng set** (Cách 2): `children_set` chứa mọi nút là con. Gốc = nút không có trong set. Bộ nhớ `O(n)`.
- **Triệt tiêu bằng số học** (Cách 3, 4): thay vì nhớ *tập* con, nhớ một **con số tổng hợp**:
  - Tổng: `S = Σ val(mọi nút) − Σ val(mọi con)` = `val(gốc)` vì mỗi nút không phải gốc được cộng một lần và trừ một lần.
  - XOR: `X = ⨁ val(mọi nút) ⨁ val(mọi con)` = `val(gốc)` vì giá trị không phải gốc xuất hiện 2 lần.
  - Sau đó tìm nút có `val` bằng con số đó (giá trị duy nhất ⇒ đúng một nút).

**Quy tắc:** *nếu* chỉ cần tìm "phần tử xuất hiện lẻ lần / thiếu một lần" *thì* nghĩ đến tổng hoặc XOR để đạt bộ nhớ `O(1)`.

## 7. Các cách tiếp theo

### Cách 2 — Tập hợp các nút con `O(n)` thời gian, `O(n)` bộ nhớ

1. Duyệt mọi nút, thêm từng `child` vào `seen` (theo đối tượng).
2. Duyệt lại, trả nút không có trong `seen`.

In [10]:
def set_root(tree):
    seen = set()
    for nd in tree:
        for ch in nd.children:
            seen.add(ch)
    for nd in tree:
        if nd not in seen:
            return nd

for e in examples:
    root, nodes = make_input(e, seed=12)
    got = set_root(nodes)
    print(got, got is root)

Node(1) True
Node(1) True


In [11]:
class SolutionSet:
    def findRoot(self, tree: list['Node']) -> 'Node':
        seen = {ch for node in tree for ch in node.children}
        for node in tree:
            if node not in seen:
                return node

root, nodes = make_input(ex2, seed=5)
print(SolutionSet().findRoot(nodes) is root)

True


Biến thể tương đương: lưu **giá trị** con (`seen_vals = {ch.val ...}`) rồi so `node.val not in seen_vals` — đúng vì giá trị duy nhất. Nhưng đừng trộn lẫn so **đối tượng** với tập **giá trị** (xem mục 13).

**Độ phức tạp:** Time `O(n)` (tổng số cạnh `n − 1`), Space `O(n)`.

### Cách 3 — Tổng giá trị (bộ nhớ `O(1)`)

1. `s = 0`. Với mỗi nút: `s += node.val`; với mỗi con: `s -= child.val`.
2. Cuối cùng `s` = giá trị gốc. Tìm nút có `val == s`.

Python có số nguyên không giới hạn nên không lo tràn số (ngôn ngữ khác có thể tràn → XOR an toàn hơn).

In [12]:
def sum_root(tree):
    s = 0
    for nd in tree:
        s += nd.val
        for ch in nd.children:
            s -= ch.val
    for nd in tree:
        if nd.val == s:
            return nd

for e in examples:
    root, nodes = make_input(e, seed=13)
    got = sum_root(nodes)
    print(got, got is root)

Node(1) True
Node(1) True


In [13]:
class SolutionSum:
    def findRoot(self, tree: list['Node']) -> 'Node':
        s = 0
        for node in tree:
            s += node.val - sum(child.val for child in node.children)
        return next(node for node in tree if node.val == s)

root, nodes = make_input(ex2, seed=6)
print(SolutionSum().findRoot(nodes) is root)

True


**Độ phức tạp:** Time `O(n)` (2 lượt quét), Space `O(1)` phụ.

### Cách 4 — XOR (lời giải doocs)

Giống Cách 3 nhưng thay `+`/`−` bằng `^`. Ưu điểm: không bao giờ tràn số (trong C++/Java), và "cộng" với "trừ" là cùng một phép.

In [14]:
def xor_root(tree):
    x = 0
    for nd in tree:
        x ^= nd.val
        for ch in nd.children:
            x ^= ch.val
    return next(nd for nd in tree if nd.val == x)

for e in examples:
    root, nodes = make_input(e, seed=14)
    got = xor_root(nodes)
    print(got, got is root, serialize(got) == e)

Node(1) True True
Node(1) True True


In [15]:
class SolutionXor:
    def findRoot(self, tree: list['Node']) -> 'Node':
        x = 0
        for node in tree:
            x ^= node.val
            for child in node.children:
                x ^= child.val
        return next(node for node in tree if node.val == x)

root, nodes = make_input(ex1, seed=7)
print(SolutionXor().findRoot(nodes) is root)

True


**Độ phức tạp:** Time `O(n)`, Space `O(1)` phụ — đáp ứng follow-up.

> Lưu ý: cả Cách 3 và 4 **dựa vào giả thiết giá trị duy nhất**. Nếu hai nút cùng giá trị, bước "tìm nút có val bằng X" có thể trả sai nút; khi đó dùng Cách 2 (theo đối tượng).

## 8. TRACE từng bước

Trace Cách 4 (XOR) trên Ví dụ 1, mảng đã xáo trộn với seed cố định. Bảng trạng thái mẫu (thứ tự `[5, 4, 3, 6, 2, 1]`):

| nút | x ^= val | children | x sau khi XOR các con |
|---|---|---|---|
| 5 | 0^5 = 5 | — | 5 |
| 4 | 5^4 = 1 | — | 1 |
| 3 | 1^3 = 2 | 5, 6 | 2^5^6 = 1 |
| 6 | 1^6 = 7 | — | 7 |
| 2 | 7^2 = 5 | — | 5 |
| 1 | 5^1 = 4 | 3, 2, 4 | 4^3^2^4 = 1 |

Kết quả `x = 1` → trả `Node(1)`.

In [16]:
def trace_xor(nodes):
    x = 0
    for nd in nodes:
        before = x
        x ^= nd.val
        mid = x
        for ch in nd.children:
            x ^= ch.val
        print(f"nút {nd.val:2d}: {before:2d} ^ {nd.val:2d} = {mid:2d} | con={[c.val for c in nd.children]} -> x={x}")
    ans = next(nd for nd in nodes if nd.val == x)
    print("=> gốc:", ans)
    return ans

root = build_nary(ex1)
by_val = {nd.val: nd for nd in all_nodes(root)}
order = [by_val[v] for v in [5, 4, 3, 6, 2, 1]]       # thứ tự như bảng
assert trace_xor(order) is root

nút  5:  0 ^  5 =  5 | con=[] -> x=5
nút  4:  5 ^  4 =  1 | con=[] -> x=1
nút  3:  1 ^  3 =  2 | con=[5, 6] -> x=1
nút  6:  1 ^  6 =  7 | con=[] -> x=7
nút  2:  7 ^  2 =  5 | con=[] -> x=5
nút  1:  5 ^  1 =  4 | con=[3, 2, 4] -> x=1
=> gốc: Node(1)


Trace Cách 3 (tổng) trên Ví dụ 2 với mảng xáo trộn seed 2024:

In [17]:
def trace_sum(nodes):
    s = 0
    for nd in nodes:
        delta = nd.val - sum(c.val for c in nd.children)
        s += delta
        print(f"nút {nd.val:2d} con={[c.val for c in nd.children]} delta={delta:+d} s={s}")
    return next(nd for nd in nodes if nd.val == s)

root, nodes = make_input(ex2, seed=2024)
got = trace_sum(nodes)
print("=> gốc:", got, got is root)

nút  1 con=[2, 3, 4, 5] delta=-13 s=-13
nút  9 con=[13] delta=-4 s=-17
nút 11 con=[14] delta=-3 s=-20
nút  2 con=[] delta=+2 s=-18
nút 13 con=[] delta=+13 s=-5
nút  6 con=[] delta=+6 s=1
nút 14 con=[] delta=+14 s=15
nút  7 con=[11] delta=-4 s=11
nút  4 con=[8] delta=-4 s=7
nút  5 con=[9, 10] delta=-14 s=-7
nút 10 con=[] delta=+10 s=3
nút 12 con=[] delta=+12 s=15
nút  3 con=[6, 7] delta=-10 s=5
nút  8 con=[12] delta=-4 s=1
=> gốc: Node(1) True


## 9. Code nộp LeetCode

Bản chính: XOR (đúng như doocs), bộ nhớ `O(1)`. Trên LeetCode lớp `Node` đã có sẵn.

In [18]:
class Solution:
    def findRoot(self, tree: list['Node']) -> 'Node':
        x = 0
        for node in tree:
            x ^= node.val
            for child in node.children:
                x ^= child.val
        return next(node for node in tree if node.val == x)

def driver(data, seed):
    """Giả lập driver LeetCode: dựng cây, xáo trộn nút, gọi findRoot, tuần tự hoá kết quả."""
    root, nodes = make_input(data, seed)
    got = Solution().findRoot(nodes)
    return got is root, serialize(got)

for e in examples:
    ok, out = driver(e, seed=99)
    assert ok and out == e
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

Cây có `n` nút và `n − 1` cạnh cha→con. Mỗi nút không phải gốc là **con** của đúng một nút (vì cây liên thông, không chu trình, có hướng từ gốc), còn gốc không là con của ai.

Xét đa tập `M` = {val của mọi nút trong `tree`} ∪ {val của mọi `child` của mọi nút}. Mỗi giá trị không phải gốc có mặt **2 lần** trong `M`, giá trị gốc có mặt **1 lần**.

- **XOR:** `⨁M` — các cặp triệt tiêu (`a ^ a = 0`), thứ tự không quan trọng (giao hoán, kết hợp) → còn lại `val(gốc)`.
- **Tổng:** cộng phần "nút", trừ phần "con" → mỗi không-gốc `+v − v = 0` → còn `val(gốc)`.

Vì giá trị là duy nhất, đúng một nút trong `tree` có `val` này, và đó chính là gốc (đối tượng). **Set:** tập `seen` chứa đúng `n − 1` nút không phải gốc → nút duy nhất ngoài `seen` là gốc. ∎

## 11. Phân tích Time / Space

Gọi `n` = số nút (`≤ 5·10⁴`); tổng độ dài các `children` = `n − 1`.

| Cách | Time | Space phụ |
|---|---|---|
| Brute force | `O(n²)` worst (ứng viên gốc ở cuối mảng) | `O(1)` |
| Set các con | `O(n)` trung bình (băm) | `O(n)` |
| Tổng | `O(n)` | `O(1)` |
| XOR | `O(n)` | `O(1)` |

Brute force có best case `O(n)` nếu gốc tình cờ ở đầu mảng, nhưng thứ tự là tuỳ ý nên phải tính worst.

## 12. Bộ test

Gồm: ví dụ đề với nhiều seed xáo trộn, cây 1 nút, cây "đường thẳng", cây "hình sao", và cây ngẫu nhiên (giá trị duy nhất, có thể âm / số lớn) đối chiếu mọi cách. Kiểm tra **trả đúng đối tượng gốc** bằng `is`.

In [19]:
METHODS = {
    "brute": brute_root,
    "set": lambda t: SolutionSet().findRoot(t),
    "sum": lambda t: SolutionSum().findRoot(t),
    "xor": lambda t: Solution().findRoot(t),
}

def check(root, nodes):
    for name, f in METHODS.items():
        got = f(list(nodes))
        assert got is root, (name, got, root)

for e in examples:
    for seed in range(20):
        check(*make_input(e, seed))

single = Node(42)
check(single, [single])

chain = [1, None] + sum(([v, None] for v in range(2, 8)), [])
chain = chain[:-1]
r, ns = make_input(chain, 3)
print("chain:", serialize(r))
check(r, ns)

star = [0, None] + list(range(1, 30))
check(*make_input(star, 4))
print("ví dụ + edge: OK")

chain: [1, None, 2, None, 3, None, 4, None, 5, None, 6, None, 7]
ví dụ + edge: OK


In [20]:
def random_tree(n, rng, neg=False):
    lo, hi = (-10**9, 10**9) if neg else (0, 5 * 10**4)
    vals = rng.sample(range(lo, hi + 1), n)
    nodes = [Node(v) for v in vals]
    for i in range(1, n):
        par = nodes[rng.randrange(i)]          # cha là một nút đã tạo trước
        par.children.append(nodes[i])
    root = nodes[0]
    shuffled = nodes[:]
    rng.shuffle(shuffled)
    return root, shuffled

rng = random.Random(1506)
for _ in range(300):
    root, nodes = random_tree(rng.randint(1, 40), rng, neg=rng.random() < 0.5)
    check(root, nodes)
    assert serialize(Solution().findRoot(nodes)) == serialize(root)
print("300 cây ngẫu nhiên: mọi cách trả đúng đối tượng gốc")

300 cây ngẫu nhiên: mọi cách trả đúng đối tượng gốc


In [21]:
root, nodes = random_tree(5 * 10**4, random.Random(77))
for name in ["set", "sum", "xor"]:
    t = time.perf_counter()
    assert METHODS[name](nodes) is root
    print(f"{name:4s}: {time.perf_counter() - t:.3f}s với n = 50000")

set : 0.029s với n = 50000


sum : 0.043s với n = 50000


xor : 0.021s với n = 50000


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Giả định `tree[0]` là gốc

Thứ tự mảng là **tuỳ ý**; trên ví dụ đã xáo trộn điều này thường sai.

In [22]:
def wrong_first(tree):
    return tree[0]                                  # SAI: thứ tự tuỳ ý

root, nodes = make_input(ex1, seed=1506)
got = wrong_first(nodes)
print("sai :", got, "là gốc?", got is root, "| đúng:", Solution().findRoot(nodes))

sai : Node(6) là gốc? False | đúng: Node(1)


### Lỗi 2 — So **đối tượng** với tập **giá trị**

`seen` chứa số nguyên, còn `node` là đối tượng `Node` → `node not in seen` luôn `True` → luôn trả nút đầu tiên.

In [23]:
def wrong_mix(tree):
    seen = {ch.val for nd in tree for ch in nd.children}
    for nd in tree:
        if nd not in seen:                          # SAI: phải là nd.val not in seen
            return nd

root, nodes = make_input(ex1, seed=1506)
got = wrong_mix(nodes)
print("sai :", got, "là gốc?", got is root, "| đúng:", Solution().findRoot(nodes))

sai : Node(6) là gốc? False | đúng: Node(1)


### Lỗi 3 — Trả về **giá trị** thay vì **nút**

Tính đúng `x` nhưng `return x` → driver nhận số nguyên, không tuần tự hoá được cây.

In [24]:
def wrong_return_val(tree):
    x = 0
    for nd in tree:
        x ^= nd.val
        for ch in nd.children:
            x ^= ch.val
    return x                                        # SAI: phải trả Node

root, nodes = make_input(ex1, seed=1506)
got = wrong_return_val(nodes)
print("sai : trả về", repr(got), "kiểu", type(got).__name__, "| là Node?", isinstance(got, Node))
print("đúng:", Solution().findRoot(nodes), type(Solution().findRoot(nodes)).__name__)

sai : trả về 1 kiểu int | là Node? False
đúng: Node(1) Node


### Lỗi 4 — Chọn nút có nhiều con nhất / có độ sâu cây con lớn nhất "bằng mắt"

Heuristic "gốc có nhiều con nhất" sai ngay khi một nút con có nhiều con hơn gốc.

In [25]:
def wrong_most_children(tree):
    return max(tree, key=lambda nd: len(nd.children))   # SAI: heuristic

data = [1, None, 2, None, 3, 4, 5, 6]           # 1 -> 2 -> {3,4,5,6}
root, nodes = make_input(data, seed=0)
got = wrong_most_children(nodes)
print("sai :", got, "| đúng:", Solution().findRoot(nodes))

sai : Node(2) | đúng: Node(1)


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Cho mọi nút/cạnh, tìm gốc"** → gốc = **bậc vào 0** (giống tìm *town judge* LC 997, tìm nguồn trong DAG).
- **"Mỗi phần tử xuất hiện 2 lần trừ một phần tử"** → XOR (LC 136 *Single Number*), hoặc tổng (LC 268 *Missing Number*).
- Follow-up "bộ nhớ O(1)" → nghĩ tới **mã hoá thông tin bằng một con số** (tổng, XOR) thay vì set/dict.
- Luôn hỏi lại: **giá trị có duy nhất không?** — quyết định có dùng được cách số học hay không.
- Test bài "trả về đối tượng": so bằng `is`, xáo trộn input với seed cố định.

## 15. Biến thể của bài

1. **Giá trị không duy nhất:** dùng set theo đối tượng (Cách 2), hoặc XOR trên `id(node)` (vẫn `O(1)` bộ nhớ!).
2. **Rừng (nhiều cây):** tìm mọi gốc = mọi nút có bậc vào 0.
3. **Cho danh sách cạnh `[cha, con]`** (LC 1557 *Minimum Number of Vertices to Reach All Nodes*): đếm bậc vào.
4. **Cây nhị phân** với mảng nút xáo trộn: y hệt, chỉ thay `children` bằng `left`, `right`.
5. **LC 136 / 268 / 389** — các bài XOR/tổng cổ điển.

In [26]:
# Biến thể 1: giá trị KHÔNG duy nhất -> XOR trên id(node) vẫn O(1) bộ nhớ
def find_root_by_id(tree):
    x = 0
    for nd in tree:
        x ^= id(nd)
        for ch in nd.children:
            x ^= id(ch)
    return next(nd for nd in tree if id(nd) == x)

dup = Node(7, [Node(7, [Node(7)]), Node(7)])     # mọi nút đều có val = 7
nodes = all_nodes(dup)
random.Random(1).shuffle(nodes)
print("XOR theo val (sai nếu trùng):", Solution().findRoot(nodes) is dup)
print("XOR theo id (đúng)        :", find_root_by_id(nodes) is dup)

XOR theo val (sai nếu trùng): False
XOR theo id (đúng)        : True


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Mỗi ứng viên quét mọi cạnh | `O(n²)` | `O(1)` | Chỉ để đối chiếu |
| 2. Set các con | Gốc ∉ tập con | `O(n)` | `O(n)` | Dễ hiểu nhất, không cần val duy nhất |
| 3. Tổng | Σ nút − Σ con = val gốc | `O(n)` | `O(1)` | Follow-up, val duy nhất |
| 4. XOR | ⨁ nút ⨁ con = val gốc | `O(n)` | `O(1)` | Follow-up, không lo tràn số |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`x = 0; for v in [6, 1, 6, 9, 1]: x ^= v` — `x` bằng bao nhiêu?

### Bài 2 (tính tay)
Với Ví dụ 2, tính `Σ val(mọi nút)` và `Σ val(mọi con)`. Hiệu bằng bao nhiêu?

### Bài 3 (điền TODO)
Viết `parent_map(tree)`: trả dict ánh xạ **nút con → nút cha** (gốc không có trong dict).

### Bài 4 (tự viết lại)
Viết `depths(tree)`: cho mảng nút xáo trộn, trả dict `val → độ sâu` (gốc có độ sâu 0). Gợi ý: tìm gốc trước.

### Bài 5 (mini challenge)
Cho mảng nút (xáo trộn) của một **rừng** gồm nhiều cây N-phân. Viết `find_roots(nodes)` trả danh sách các gốc, sắp theo `val`.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def parent_map(tree):
    # TODO: duyệt mọi nút, với mỗi con gán par[child] = node
    pass

# Bài 4 — TODO
def depths(tree):
    # TODO: root = Solution().findRoot(tree); BFS từ root lưu độ sâu
    pass

# Bài 5 — TODO
def find_roots(nodes):
    # TODO: tập các con; gốc = nút không thuộc tập; sort theo val
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Cặp giống nhau triệt tiêu. (2) 6 và 1 đều xuất hiện 2 lần. (3) Còn lại 9.

**Bài 2.** (1) Σ val mọi nút = 1 + 2 + … + 14. (2) Mọi nút trừ gốc đều là con của ai đó. (3) Hiệu = val gốc = 1.

**Bài 3.** (1) Dùng dict với khoá là đối tượng nút. (2) Một vòng lặp lồng `for nd in tree: for ch in nd.children`. (3) `par[ch] = nd`.

**Bài 4.** (1) Tìm gốc bằng XOR. (2) BFS hàng đợi `(nút, độ sâu)`. (3) Con có độ sâu cha + 1.

**Bài 5.** (1) Không dùng được XOR (nhiều gốc trộn lẫn). (2) Dùng set các con như Cách 2. (3) `sorted(roots, key=lambda nd: nd.val)`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
x = 0
for v in [6, 1, 6, 9, 1]:
    x ^= v
print("Bài 1:", x)
assert x == 9

# Bài 2
r2 = build_nary(ex2)
ns2 = all_nodes(r2)
s_nodes = sum(nd.val for nd in ns2)
s_child = sum(ch.val for nd in ns2 for ch in nd.children)
print("Bài 2:", s_nodes, s_child, s_nodes - s_child)
assert s_nodes - s_child == 1 and s_nodes == sum(range(1, 15))

Bài 1: 9
Bài 2: 105 104 1


In [29]:
# Bài 3
def parent_map_sol(tree):
    par = {}
    for nd in tree:
        for ch in nd.children:
            par[ch] = nd
    return par

root, nodes = make_input(ex1, seed=8)
pm = parent_map_sol(nodes)
assert root not in pm and len(pm) == len(nodes) - 1
assert {ch.val: p.val for ch, p in pm.items()} == {3: 1, 2: 1, 4: 1, 5: 3, 6: 3}

# Bài 4
def depths_sol(tree):
    root = Solution().findRoot(tree)
    res, q = {}, deque([(root, 0)])
    while q:
        nd, d = q.popleft()
        res[nd.val] = d
        for ch in nd.children:
            q.append((ch, d + 1))
    return res

assert depths_sol(nodes) == {1: 0, 3: 1, 2: 1, 4: 1, 5: 2, 6: 2}
root2, nodes2 = make_input(ex2, seed=8)
assert depths_sol(nodes2)[14] == 4
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [30]:
# Bài 5
def find_roots_sol(nodes):
    kids = {ch for nd in nodes for ch in nd.children}
    return sorted((nd for nd in nodes if nd not in kids), key=lambda nd: nd.val)

rng = random.Random(5)
for _ in range(100):
    k = rng.randint(1, 4)
    roots, pool = [], []
    for _ in range(k):
        r, ns = random_tree(rng.randint(1, 10), rng)
        roots.append(r)
        pool.extend(ns)
    # random_tree có thể trùng giá trị giữa các cây; find_roots theo đối tượng nên vẫn đúng
    rng.shuffle(pool)
    got = find_roots_sol(pool)
    assert set(map(id, got)) == set(map(id, roots))
    assert [g.val for g in got] == sorted(r.val for r in roots)
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "phần tử xuất hiện lẻ lần" bằng XOR:**

```python
x = 0
for node in tree:
    x ^= node.val
    for child in node.children:
        x ^= child.val
return next(node for node in tree if node.val == x)
```

**Template bậc vào 0 (không cần val duy nhất):**

```python
seen = {ch for node in tree for ch in node.children}
return next(node for node in tree if node not in seen)
```

**Checklist trước khi submit:**
- [ ] Trả về **đối tượng `Node`**, không phải `val`.
- [ ] Không giả định thứ tự mảng (`tree[0]` không chắc là gốc).
- [ ] So đối tượng với tập đối tượng, giá trị với tập giá trị — không trộn.
- [ ] Dùng tổng/XOR chỉ khi giá trị **duy nhất**.
- [ ] Test bằng cách xáo trộn mảng nút với seed cố định và kiểm tra `is`.
- [ ] Đúng chữ ký: `findRoot(self, tree: list['Node']) -> 'Node'`.